In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

%matplotlib inline

In [ ]:
# Load the data and create a vocabulary of characters and words
root = Path('..')
data_path = root / 'data' / 'shakespeare.txt'
text = open(data_path, 'r', encoding='utf-8').read()
print(f'Dataset length in characters: {len(text)}')

In [ ]:
# Create a vocabulary of characters
chars = sorted(list(set(text)))
VOCAB_SIZE = len(chars)
print(f'Vocabulary size: {VOCAB_SIZE}')
print(''.join(chars))

In [ ]:
# Tokenize the text
char_to_index = { ch:i for i, ch in enumerate(chars) }
index_to_char = { i:ch for i, ch in enumerate(chars) }

encode = lambda s: [char_to_index[ch] for ch in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([index_to_char[i] for i in l]) # decoder: take a list of integers, output a string

# example of encoding and decoding
text_example = 'Hello there!'
print(f'Encode: {encode(text_example)}')
print(f'Decode: {decode(encode(text_example))}')

In [ ]:
# Encode the text and convert it to a tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape)

In [ ]:
# Split the data into training, validation and test sets
def split_data(data, train_frac=0.9, val_frac=0.1):
    n = len(data)
    train_data = data[:int(n*train_frac)]
    val_data = data[int(n*train_frac):int(n*(train_frac+val_frac))]
    return train_data, val_data

train_data, val_data = split_data(data)
print(f'Train data size: {len(train_data)}')
print(f'Validation data size: {len(val_data)}')

In [ ]:
# Data loader - get a batch of data
BATCH_SIZE = 4 # how many inpendent sequences to process in parallel
BLOCK_SIZE = 8 # what is the maxumum cotext lenght to predict

torch.manual_seed(42) # for reproducibility

def get_batch(split):
    match split:
        case 'train': data = train_data
        case 'val': data = val_data
        case _: raise ValueError('Invalid split')

    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,)) # Randomly select starting indices for sequences
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix]) # Extract input sequences (x) from those indices and stack them
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix]) # Extract target sequences (y) with +1 offset (next character prediction) and stack them
    return x, y

Xb, Yb = get_batch('train')
print(f'Input batch shape: {Xb.shape},\n data: {Xb}\n')
print(f'Target batch shape: {Yb.shape},\n data: {Yb}\n')

for b in range(BATCH_SIZE): # batch dimension
    for t in range(BLOCK_SIZE): # time dimension
        context = Xb[b, :t+1]
        target = Yb[b, t]
        print(f'When input is {context.tolist()}, the target character is {target}')

In [ ]:
# Define the model
# Bigram Language Model as a baseline

class BigramLM(nn.Module):

    def __init__(self, VOCAB_SIZE):
        super().__init__()
        self.token_embedding_table = nn.Embedding(VOCAB_SIZE, VOCAB_SIZE)
    
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) 
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape # (BATCH_SIZE, BLOCK_SIZE, VOCAB_SIZE)
            logits = logits.view(B * T, C)
            targets = targets.view(B * T) # or view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx: (BATCH_SIZE, BLOCK_SIZE) arrays of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # get the predictions
            logits = logits[:, -1, :] # becomes (BATCH_SIZE, VOCAB_SIZE)
            probs = F.softmax(logits, dim=-1) # apply softmax to get probabilities
            idx_next = torch.multinomial(probs, num_samples=1) # sample from the distribution
            idx = torch.cat([idx, idx_next], dim=1) # add the new token to the context (BATCH_SIZE, BLOCK_SIZE+1)
        return idx

model = BigramLM(VOCAB_SIZE)
logits, loss = model(Xb, Yb)
print(logits.shape, loss)
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

In [ ]:
# Create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
# Training loop
EPOCHS = 10_000
BATCH_SIZE = 32

for epoch in range(EPOCHS):

    Xb, Yb = get_batch('train') # sample a batch of data

    # evaluate the loss
    logits, loss = model(Xb, Yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if epoch % (EPOCHS / 10) == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

In [ ]:
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

In [ ]:
# The mathematical trick in Self-Attention

B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
print(x.shape)

In [ ]:
# Example 1.0: we want x[b, t] = mean_{i<=t} x[b, i]
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0)

In [ ]:
# Example 1.1: Simple example of how to do the same with matrix multiplication
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, dim=1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print(f'A:\n{a}\nB:\n{b}\nC:\n{c}')

In [ ]:
# Example 2.0: rewriting example 1 with matrix multiplication
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) -> (B, T, C)
torch.allclose(xbow, xbow2)

In [ ]:
# Example 3.0: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

In [ ]:
# Example 4.0: Self-Attention
B, T, C = 4, 8, 32 # Batch, Time, Channels
x = torch.randn(B, T, C)

# Single head self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, head_size)
q = query(x) # (B, T, head_size)
wei = q @ k.transpose(-2, -1) / (head_size ** 0.5) # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x
print(out.shape)

In [ ]:
# Generation from checkpoint model

from gpt_lecture.gpt import GPTLM

ROOT = Path('..')
checkpoint_path = ROOT / 'models' / 'checkpoint.pth'

DATA_PATH = ROOT / 'data' / 'shakespeare.txt'
with open(DATA_PATH, 'r') as f:
    text = f.read()

chars = sorted(list(set(text)))
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
encode = lambda x: torch.tensor([char_to_int[c] for c in x], dtype=torch.long)
decode = lambda x: ''.join([int_to_char[int(c)] for c in x])

BATCH_SIZE = 32
BLOCK_SIZE = 128
N_EMBD = 16
MAX_ITERS = 100
VOCAB_SIZE = len(chars)
EVAL_ITERS = MAX_ITERS // 100
LEARNING_RATE = 5e-4
N_HEAD = 4
N_LAYER = 4
DROPOUT = 0.2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')

model = GPTLM(
    block_size=BLOCK_SIZE,
    vocab_size=VOCAB_SIZE,
    n_embd=N_EMBD,
    n_head=N_HEAD,
    n_layer=N_LAYER,
    dropout=DROPOUT
)
model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
model.eval()
model.to(DEVICE)


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)
print(f'Generated text: {decode(model.generate(context, max_new_tokens=500)[0].tolist())}')